In [109]:
import numpy as np
import pandas as pd
import torch
import folium
from folium.plugins import MeasureControl
from IPython.display import display

from lstm import lstm_seq2seq

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [110]:
feature_cols = [
    "LAT",
    "LON",
    "SPEED",
    "dt",
    "cog_sin",
    "cog_cos",
    "hdg_sin",
    "hdg_cos",
    "dlat",
    "dlon",
]

target_cols = ["dlat", "dlon"]

SEQ_LEN = 10
TARGET_LEN = 1

HIDDEN_SIZE = 128
NUM_LAYERS = 2
DROPOUT = 0.2

In [111]:
df_test = pd.read_csv("../data/vessel_prediction_model_data/mid_model_data/mid_test_data.csv")
df_train = pd.read_csv("../data/vessel_prediction_model_data/mid_model_data/mid_train_data.csv")

In [112]:
feature_means = df_train[feature_cols].mean()
feature_stds  = df_train[feature_cols].std()

target_means = df_train[target_cols].mean()
target_stds  = df_train[target_cols].std()

In [113]:
df_test[feature_cols] = (df_test[feature_cols] - feature_means) / feature_stds
df_test[target_cols]  = (df_test[target_cols]  - target_means) / target_stds

In [114]:
def make_windows_from_df(df, feature_cols, target_cols, seq_len=5, target_len=1,
                         route_col="voyage_id", time_col="TIME"):
    X_list = []
    Y_list = []
    meta_list = []

    for route_id, group in df.groupby(route_col):
        group = group.sort_values(time_col).reset_index(drop=True)

        feat = group[feature_cols].to_numpy(dtype=np.float32)
        targ = group[target_cols].to_numpy(dtype=np.float32)

        n = len(group)
        window_count = n - seq_len - target_len + 1
        if window_count <= 0:
            continue

        for i in range(window_count):
            x_win = feat[i:i+seq_len]
            y_win = targ[i+seq_len:i+seq_len+target_len]

            X_list.append(x_win)
            Y_list.append(y_win)

            meta_list.append({
                "voyage_id": route_id,
                "start_idx": i,
                "last_known_time": group.iloc[i + seq_len][time_col],
            })

    if len(X_list) == 0:
        raise ValueError("No windows were created.")

    X = np.stack(X_list, axis=0)   # (N, seq_len, features)
    Y = np.stack(Y_list, axis=0)   # (N, target_len, 2)

    X = np.transpose(X, (1, 0, 2)) # (seq_len, N, features)
    Y = np.transpose(Y, (1, 0, 2)) # (target_len, N, 2)

    X = torch.tensor(X, dtype=torch.float32)
    Y = torch.tensor(Y, dtype=torch.float32)

    return X, Y, meta_list

In [115]:
X_test, Y_test, meta_test = make_windows_from_df(
    df_test,
    feature_cols,
    target_cols,
    seq_len=SEQ_LEN,
    target_len=TARGET_LEN,
    route_col="voyage_id",
    time_col="TIME"
)

In [116]:
X_test = torch.tensor(X_test, dtype=torch.float32)
Y_test = torch.tensor(Y_test, dtype=torch.float32)

C:\Users\logan\AppData\Local\Temp\ipykernel_35108\2696808717.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_test = torch.tensor(X_test, dtype=torch.float32)
C:\Users\logan\AppData\Local\Temp\ipykernel_35108\2696808717.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  Y_test = torch.tensor(Y_test, dtype=torch.float32)


In [117]:
model = lstm_seq2seq(
    input_size=len(feature_cols),
    hidden_size=HIDDEN_SIZE,
    output_size=2,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
    target_indices=(0, 1),
    decoder_feature_indices=tuple(range(len(feature_cols))),
    predict_deltas=True,
).to(DEVICE)

model.load_state_dict(torch.load("best_lstm_seq2seq_dlat_dlon.pt", map_location=DEVICE))
model.eval()

C:\Users\logan\AppData\Local\Temp\ipykernel_35108\1553603474.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("best_lstm_seq2seq_dlat_dl

lstm_seq2seq(
  (encoder): lstm_encoder(
    (lstm): LSTM(10, 128, num_layers=2, dropout=0.2)
  )
  (decoder): lstm_decoder(
    (lstm): LSTM(10, 128, num_layers=2, dropout=0.2)
    (linear): Linear(in_features=128, out_features=2, bias=True)
  )
  (target_to_decoder): Linear(in_features=2, out_features=10, bias=True)
)

In [118]:
with torch.no_grad():
    Y_pred = model.predict(
        X_test.to(DEVICE),
        target_len=TARGET_LEN,
        return_absolute=False
    )

In [119]:
def dead_reckoning_next_point(lat_deg, lon_deg, speed_knots, cog_deg, dt_seconds):
    """
    Predict one next AIS point assuming constant speed and constant course.
    Returns predicted (lat, lon) in degrees.
    """
    # distance traveled in meters
    dist_m = speed_knots * 0.514444 * dt_seconds

    # bearings: 0° = north, 90° = east
    theta = np.radians(cog_deg)

    # north/east displacement
    d_north = dist_m * np.cos(theta)
    d_east  = dist_m * np.sin(theta)

    # convert meters to degrees
    dlat = d_north / 111320.0
    dlon = d_east / (111320.0 * np.cos(np.radians(lat_deg)))

    pred_lat = lat_deg + dlat
    pred_lon = lon_deg + dlon
    return pred_lat, pred_lon

In [120]:
# =========================
# DEAD RECKONING BASELINE ON TEST SET
# =========================

# convert test tensors to numpy
X_test_np = X_test.detach().cpu().numpy().copy()
Y_test_np = Y_test.detach().cpu().numpy().copy()

# feature index lookup
idx_lat = feature_cols.index("LAT")
idx_lon = feature_cols.index("LON")
idx_speed = feature_cols.index("SPEED")
idx_dt = feature_cols.index("dt")
idx_cog_sin = feature_cols.index("cog_sin")
idx_cog_cos = feature_cols.index("cog_cos")

# rebuild absolute history features from scaled X_test
X_test_unscaled = X_test_np.copy()

for col in feature_cols:
    j = feature_cols.index(col)
    X_test_unscaled[:, :, j] = X_test_unscaled[:, :, j] * feature_stds[col] + feature_means[col]

# rebuild true dlat/dlon from scaled Y_test
Y_true_delta = Y_test_np.copy()
Y_true_delta[..., 0] = Y_true_delta[..., 0] * target_stds["dlat"] + target_means["dlat"]
Y_true_delta[..., 1] = Y_true_delta[..., 1] * target_stds["dlon"] + target_means["dlon"]

# last history point for each sample
last_lat = X_test_unscaled[-1, :, idx_lat]
last_lon = X_test_unscaled[-1, :, idx_lon]
last_speed = X_test_unscaled[-1, :, idx_speed]
last_dt = X_test_unscaled[-1, :, idx_dt]
last_cog_sin = X_test_unscaled[-1, :, idx_cog_sin]
last_cog_cos = X_test_unscaled[-1, :, idx_cog_cos]

# course in degrees from sin/cos
last_cog_rad = np.arctan2(last_cog_sin, last_cog_cos)
last_cog_deg = (np.degrees(last_cog_rad) + 360.0) % 360.0

# true absolute next point from true delta
true_future_abs = np.zeros((1, X_test_unscaled.shape[1], 2), dtype=np.float64)
true_future_abs[0, :, 0] = last_lat + Y_true_delta[0, :, 0]
true_future_abs[0, :, 1] = last_lon + Y_true_delta[0, :, 1]

# dead reckoning predicted next point
pred_future_abs_dr = np.zeros((1, X_test_unscaled.shape[1], 2), dtype=np.float64)

for i in range(X_test_unscaled.shape[1]):
    pred_lat, pred_lon = dead_reckoning_next_point(
        lat_deg=last_lat[i],
        lon_deg=last_lon[i],
        speed_knots=last_speed[i],
        cog_deg=last_cog_deg[i],
        dt_seconds=last_dt[i],
    )
    pred_future_abs_dr[0, i, 0] = pred_lat
    pred_future_abs_dr[0, i, 1] = pred_lon

In [121]:
def unscale_deltas(delta_tensor, target_means, target_stds, device=None):
    # ensure tensor
    if not torch.is_tensor(delta_tensor):
        delta_tensor = torch.tensor(delta_tensor, dtype=torch.float32, device=device)
    else:
        delta_tensor = delta_tensor.to(dtype=torch.float32, device=device)

    means = torch.tensor(
        [float(target_means["dlat"]), float(target_means["dlon"])],
        dtype=torch.float32,
        device=delta_tensor.device
    ).view(1, 1, 2)

    stds = torch.tensor(
        [float(target_stds["dlat"]), float(target_stds["dlon"])],
        dtype=torch.float32,
        device=delta_tensor.device
    ).view(1, 1, 2)

    return delta_tensor * stds + means


def get_last_known_abs_point(X_test, sample_idx, feature_means, feature_stds, feature_cols):
    lat_idx = feature_cols.index("LAT")
    lon_idx = feature_cols.index("LON")

    last_lat_scaled = X_test[-1, sample_idx, lat_idx].item()
    last_lon_scaled = X_test[-1, sample_idx, lon_idx].item()

    last_lat = last_lat_scaled * float(feature_stds["LAT"]) + float(feature_means["LAT"])
    last_lon = last_lon_scaled * float(feature_stds["LON"]) + float(feature_means["LON"])

    return last_lat, last_lon


def delta_to_absolute(last_lat, last_lon, dlat, dlon):
    return last_lat + dlat, last_lon + dlon

In [122]:
def metrics_in_yards(pred_scaled, true_scaled, X_tensor,
                     feature_means, feature_stds,
                     target_means, target_stds,
                     feature_cols):

    # force everything to torch
    if not torch.is_tensor(pred_scaled):
        pred_scaled = torch.tensor(pred_scaled, dtype=torch.float32)
    else:
        pred_scaled = pred_scaled.to(dtype=torch.float32)

    if not torch.is_tensor(true_scaled):
        true_scaled = torch.tensor(true_scaled, dtype=torch.float32)
    else:
        true_scaled = true_scaled.to(dtype=torch.float32)

    if not torch.is_tensor(X_tensor):
        X_tensor = torch.tensor(X_tensor, dtype=torch.float32)
    else:
        X_tensor = X_tensor.to(dtype=torch.float32)

    device = pred_scaled.device

    true_scaled = true_scaled.to(device)
    X_tensor = X_tensor.to(device)

    pred_deg = unscale_deltas(pred_scaled, target_means, target_stds, device=device)
    true_deg = unscale_deltas(true_scaled, target_means, target_stds, device=device)

    err_deg = pred_deg - true_deg
    err_dlat_deg = err_deg[:, :, 0]
    err_dlon_deg = err_deg[:, :, 1]

    lat_idx = feature_cols.index("LAT")
    last_lat_scaled = X_tensor[-1, :, lat_idx]
    last_lat_deg = last_lat_scaled * float(feature_stds["LAT"]) + float(feature_means["LAT"])
    last_lat_deg = last_lat_deg.unsqueeze(0).expand(err_dlat_deg.shape[0], -1)

    meters_per_deg_lat = 111320.0
    meters_per_deg_lon = 111320.0 * torch.cos(torch.deg2rad(last_lat_deg))

    err_north_m = err_dlat_deg * meters_per_deg_lat
    err_east_m  = err_dlon_deg * meters_per_deg_lon

    err_m = torch.sqrt(err_north_m**2 + err_east_m**2)
    err_yards = err_m * 1.0936133

    mse_yards2 = torch.mean(err_yards ** 2).item()
    rmse_yards = torch.sqrt(torch.mean(err_yards ** 2)).item()
    mae_yards = torch.mean(err_yards).item()

    return mse_yards2, rmse_yards, mae_yards

In [123]:
mse_yards2, rmse_yards, mae_yards = metrics_in_yards(
    Y_pred,
    Y_test,
    X_test,
    feature_means,
    feature_stds,
    target_means,
    target_stds,
    feature_cols
)

print("MSE (yards^2):", mse_yards2)
print("RMSE (yards):", rmse_yards)
print("MAE (yards):", mae_yards)

MSE (yards^2): 5304786432.0
RMSE (yards): 72833.9609375
MAE (yards): 37581.50390625


In [124]:
YARDS_PER_METER = 1.0936132983377078

def haversine_m(lat1, lon1, lat2, lon2):
    R = 6371000.0
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0)**2
    c = 2.0 * np.arcsin(np.sqrt(a))
    return R * c

def metrics_yards(pred_abs, true_abs):
    err_m = haversine_m(
        true_abs[..., 0],
        true_abs[..., 1],
        pred_abs[..., 0],
        pred_abs[..., 1],
    )
    err_yd = err_m * YARDS_PER_METER

    mae_yd = np.mean(np.abs(err_yd))
    mse_yd2 = np.mean(err_yd ** 2)
    rmse_yd = np.sqrt(mse_yd2)

    return mae_yd, mse_yd2, rmse_yd, err_yd

In [125]:
# metrics in yards
mae_yd_dr, mse_yd2_dr, rmse_yd_dr, err_yd_dr = metrics_yards(pred_future_abs_dr, true_future_abs)

print("Dead Reckoning baseline metrics in yards")
print(f"MAE  (yd): {mae_yd_dr:,.3f}")
print(f"MSE  (yd²): {mse_yd2_dr:,.3f}")
print(f"RMSE (yd): {rmse_yd_dr:,.3f}")

Dead Reckoning baseline metrics in yards
MAE  (yd): 78,212.599
MSE  (yd²): 12,431,877,421.066
RMSE (yd): 111,498.329


In [126]:
def map_single_prediction(sample_idx, X_test, Y_test, Y_pred,
                          meta_test, feature_means, feature_stds,
                          target_means, target_stds, feature_cols):
    # make sure tensors
    if not torch.is_tensor(X_test):
        X_test_local = torch.tensor(X_test, dtype=torch.float32)
    else:
        X_test_local = X_test.detach().cpu()

    if not torch.is_tensor(Y_test):
        Y_test_local = torch.tensor(Y_test, dtype=torch.float32)
    else:
        Y_test_local = Y_test.detach().cpu()

    if not torch.is_tensor(Y_pred):
        Y_pred_local = torch.tensor(Y_pred, dtype=torch.float32)
    else:
        Y_pred_local = Y_pred.detach().cpu()


    # unscale deltas
    Y_pred_unscaled = unscale_deltas(Y_pred_local, target_means, target_stds).cpu()
    Y_test_unscaled = unscale_deltas(Y_test_local, target_means, target_stds).cpu()

    # last known absolute point
    last_lat, last_lon = get_last_known_abs_point(
        X_test_local, sample_idx, feature_means, feature_stds, feature_cols
    )

    # indices
    speed_idx = feature_cols.index("SPEED")
    dt_idx    = feature_cols.index("dt")
    cog_sin_idx = feature_cols.index("cog_sin")
    cog_cos_idx = feature_cols.index("cog_cos")

    # get last timestep (still scaled)
    last_speed_scaled = X_test[-1, sample_idx, speed_idx].item()
    last_dt_scaled    = X_test[-1, sample_idx, dt_idx].item()
    last_cog_sin      = X_test[-1, sample_idx, cog_sin_idx].item()
    last_cog_cos      = X_test[-1, sample_idx, cog_cos_idx].item()

    # unscale SPEED and dt
    speed = last_speed_scaled * float(feature_stds["SPEED"]) + float(feature_means["SPEED"])
    dt    = last_dt_scaled * float(feature_stds["dt"]) + float(feature_means["dt"])

    # DR prediction
    dr_lat, dr_lon = dead_reckoning_point(
        last_lat, last_lon,
        speed, dt,
        last_cog_sin, last_cog_cos
    )
    
    # target_len=1, so use [0, sample_idx]
    pred_dlat = Y_pred_unscaled[0, sample_idx, 0].item()
    pred_dlon = Y_pred_unscaled[0, sample_idx, 1].item()

    true_dlat = Y_test_unscaled[0, sample_idx, 0].item()
    true_dlon = Y_test_unscaled[0, sample_idx, 1].item()

    pred_lat, pred_lon = delta_to_absolute(last_lat, last_lon, pred_dlat, pred_dlon)
    true_lat, true_lon = delta_to_absolute(last_lat, last_lon, true_dlat, true_dlon)

    voyage_id = meta_test[sample_idx]["voyage_id"] if meta_test is not None else "unknown"
    last_time = meta_test[sample_idx].get("last_known_time", "unknown") if meta_test is not None else "unknown"

    center_lat = np.mean([last_lat, pred_lat, true_lat])
    center_lon = np.mean([last_lon, pred_lon, true_lon])

    m = folium.Map(location=[center_lat, center_lon], zoom_start=9)

    m.add_child(MeasureControl(
    primary_length_unit='miles',
    secondary_length_unit='yards'
    ))
    # Last known AIS point
    folium.CircleMarker(
        location=[last_lat, last_lon],
        radius=7,
        color="blue",
        fill=True,
        fill_color="blue",
        fill_opacity=0.9,
        popup=f"Voyage: {voyage_id}<br>Last known AIS<br>Time: {last_time}<br>Lat: {last_lat:.6f}<br>Lon: {last_lon:.6f}",
        tooltip="Last known AIS"
    ).add_to(m)

    # Predicted point (RED)
    folium.CircleMarker(
        location=[pred_lat, pred_lon],
        radius=7,
        color="red",
        fill=True,
        fill_color="red",
        fill_opacity=0.9,
        popup=f"Voyage: {voyage_id}<br>Predicted point<br>Lat: {pred_lat:.6f}<br>Lon: {pred_lon:.6f}",
        tooltip="Predicted point"
    ).add_to(m)

    # Actual point (GREEN)
    folium.CircleMarker(
        location=[true_lat, true_lon],
        radius=7,
        color="green",
        fill=True,
        fill_color="green",
        fill_opacity=0.9,
        popup=f"Voyage: {voyage_id}<br>Actual point<br>Lat: {true_lat:.6f}<br>Lon: {true_lon:.6f}",
        tooltip="Actual point"
    ).add_to(m)

    # Dead Reckoning point (ORANGE)
    folium.CircleMarker(
    location=[dr_lat, dr_lon],
    radius=7,
    color="orange",
    fill=True,
    fill_color="orange",
    fill_opacity=0.9,
    popup=f"Voyage: {voyage_id}<br>Dead Reckoning<br>Lat: {dr_lat:.6f}<br>Lon: {dr_lon:.6f}",
    tooltip="Dead Reckoning"
    ).add_to(m)

    # lines from last known point
    folium.PolyLine(
        [(last_lat, last_lon), (pred_lat, pred_lon)],
        weight=2,
        color="red",
        opacity=0.8,
        tooltip="Predicted path"
    ).add_to(m)

    folium.PolyLine(
        [(last_lat, last_lon), (true_lat, true_lon)],
        weight=2,
        color="green",
        opacity=0.8,
        tooltip="Actual path"
    ).add_to(m)

    folium.PolyLine(
    [(last_lat, last_lon), (dr_lat, dr_lon)],
    weight=2,
    color="orange",
    opacity=0.8,
    tooltip="Dead Reckoning path"
    ).add_to(m)

    return m

In [127]:
from ipywidgets import interact, IntSlider
from IPython.display import display

def show_prediction(sample_idx):
    print(f"Sample index: {sample_idx}")
    if meta_test is not None:
        print("Voyage:", meta_test[sample_idx]["voyage_id"])
        print("Last known time:", meta_test[sample_idx].get("last_known_time", "unknown"))

    m = map_single_prediction(
        sample_idx,
        X_test,
        Y_test,
        Y_pred,
        meta_test,
        feature_means,
        feature_stds,
        target_means,
        target_stds,
        feature_cols
    )
    display(m)

interact(
    show_prediction,
    sample_idx=IntSlider(min=0, max=X_test.shape[1] - 1, step=1, value=0)
)

interactive(children=(IntSlider(value=0, description='sample_idx', max=2100), Output()), _dom_classes=('widget…

<function __main__.show_prediction(sample_idx)>